# NGLab Tutorial #2: Rust OrderBook Basics

In this notebook, we'll explore NGLab's **Central Limit Order Book (CLOB)**—the high-performance Rust engine that powers order matching.

## Learning Objectives

1. Understand the OrderBook data structure
2. Execute market and limit orders
3. Visualize the bid-ask ladder
4. Calculate risk metrics (VaR)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")

# Try to import the Rust OrderBook
try:
    from nglab import OrderBook, Order, OrderType
    RUST_AVAILABLE = True
    print("✓ Rust OrderBook imported successfully")
except ImportError:
    RUST_AVAILABLE = False
    print("⚠ Rust module not available. Using Python simulation instead.")
    print("  To build: cd rust && maturin develop")

## 1. OrderBook Architecture

### Core Data Structures

```rust
pub struct OrderBook {
    bids: IndexMap<i64, PriceLevel>,  // Buy orders (price * 10,000)
    asks: IndexMap<i64, PriceLevel>,  // Sell orders
}

pub struct PriceLevel {
    price: f64,
    orders: VecDeque<Order>,  // FIFO queue
}
```

### Why Fixed-Point Arithmetic?

Storing prices as integers (`price * 10_000`) avoids floating-point precision issues:

```python
# Floating-point comparison issues
assert 0.1 + 0.2 == 0.3  # False! (0.30000000000000004)

# Fixed-point solution
assert (1000 + 2000) == 3000  # True
```

## 2. Creating an OrderBook

Let's create a simple BTC/USD orderbook and add some orders:

In [ ]:
if RUST_AVAILABLE:
    # Create OrderBook instance
    ob = OrderBook()
    
    # Add buy (bid) orders
    ob.add_order(Order(
        order_type=OrderType.Limit,
        side="BUY",
        price=50000.0,
        quantity=0.5,
        timestamp=0
    ))
    
    ob.add_order(Order(
        order_type=OrderType.Limit,
        side="BUY",
        price=49950.0,
        quantity=1.0,
        timestamp=1
    ))
    
    # Add sell (ask) orders
    ob.add_order(Order(
        order_type=OrderType.Limit,
        side="SELL",
        price=50050.0,
        quantity=0.8,
        timestamp=2
    ))
    
    print("OrderBook created with 3 limit orders")
    print(f"Best Bid: ${ob.best_bid():.2f}")
    print(f"Best Ask: ${ob.best_ask():.2f}")
    print(f"Spread: ${ob.spread():.2f}")
else:
    # Python-based simulation
    print("Creating mock orderbook data...")
    
    best_bid = 50000.0
    best_ask = 50050.0
    spread = best_ask - best_bid
    
    print(f"Best Bid: ${best_bid:.2f}")
    print(f"Best Ask: ${best_ask:.2f}")
    print(f"Spread: ${spread:.2f}")

## 3. Order Matching Mechanics

### Price-Time Priority (FIFO)

Orders are matched based on:
1. **Price**: Best price gets matched first
2. **Time**: Among equal prices, oldest order gets priority

Example:
```
Bids:  100 @ $50.00 (t=10:00:01)
        50 @ $50.00 (t=10:00:02)
       200 @ $49.95

Incoming: SELL 120 @ Market

Match 1: 100 @ $50.00 (fills first order)
Match 2:  20 @ $50.00 (partial fill second order)
```

In [ ]:
if RUST_AVAILABLE:
    # Execute a market sell order
    market_order = Order(
        order_type=OrderType.Market,
        side="SELL",
        price=0.0,  # Market orders don't specify price
        quantity=0.3,
        timestamp=3
    )
    
    trades = ob.match_order(market_order)
    
    print(f"\nExecuted {len(trades)} trade(s):")
    for i, trade in enumerate(trades, 1):
        print(f"  Trade {i}: {trade.quantity:.4f} BTC @ ${trade.price:.2f}")
else:
    print("\nSimulated trade execution:")
    print("  Trade 1: 0.3000 BTC @ $50000.00")

## 4. Visualizing the Order Book Ladder

In [ ]:
# Generate sample order book data for visualization
np.random.seed(42)

# Create realistic bid/ask ladder
mid_price = 50000.0
n_levels = 10

# Bids (decreasing from mid)
bid_prices = mid_price - np.array([10, 20, 30, 50, 75, 100, 150, 200, 250, 300])
bid_volumes = np.random.exponential(scale=0.5, size=n_levels) + 0.1

# Asks (increasing from mid)
ask_prices = mid_price + np.array([10, 20, 35, 55, 80, 110, 160, 210, 260, 310])
ask_volumes = np.random.exponential(scale=0.5, size=n_levels) + 0.1

# Create DataFrame
ladder_df = pd.DataFrame({
    'Bid Price': bid_prices,
    'Bid Volume': bid_volumes,
    'Ask Price': ask_prices,
    'Ask Volume': ask_volumes
})

print("Order Book Ladder:")
print(ladder_df.head())

In [ ]:
# Visualize the order book
fig, ax = plt.subplots(figsize=(12, 6))

# Plot bids (green)
ax.barh(bid_prices, bid_volumes, height=8, color='green', alpha=0.6, label='Bids')

# Plot asks (red)
ax.barh(ask_prices, -ask_volumes, height=8, color='red', alpha=0.6, label='Asks')

# Mid price line
ax.axhline(y=mid_price, color='black', linestyle='--', linewidth=2, label='Mid Price')

ax.set_xlabel('Volume (BTC)', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.set_title('BTC/USD Order Book Ladder', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMarket Depth: {ladder_df['Bid Volume'].sum():.2f} BTC (bids), {ladder_df['Ask Volume'].sum():.2f} BTC (asks)")

## 5. Risk Management: Value at Risk (VaR)

VaR answers: *"What is the maximum probable loss over the next period?"*

NGLab uses **Historical Simulation**:

```python
returns = np.diff(prices) / prices[:-1]
sorted_returns = np.sort(returns)
index = int((1 - confidence) * len(returns))
var = -sorted_returns[index]
```

In [ ]:
# Generate synthetic price history
np.random.seed(123)
days = 252  # 1 trading year
drift = 0.0005  # 5bps daily drift
volatility = 0.02  # 2% daily vol

returns = np.random.normal(drift, volatility, days)
prices = 50000 * np.exp(np.cumsum(returns))

# Calculate VaR at 95% confidence
confidence = 0.95
sorted_returns = np.sort(returns)
var_index = int((1 - confidence) * len(returns))
var_95 = -sorted_returns[var_index]

# For a $100,000 portfolio
portfolio_value = 100000
var_dollar = portfolio_value * var_95

print(f"\n=== Value at Risk (95% confidence) ===")
print(f"VaR (percentage): {var_95*100:.2f}%")
print(f"VaR (dollars): ${var_dollar:,.2f}")
print(f"\nInterpretation: We are 95% confident that daily losses will not exceed ${var_dollar:,.2f}")

In [ ]:
# Visualize return distribution and VaR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Price evolution
ax1.plot(prices, linewidth=2, color='steelblue')
ax1.set_title('BTC Price Simulation (252 days)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Day')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)

# Return distribution
ax2.hist(returns, bins=50, color='coral', alpha=0.7, edgecolor='black')
ax2.axvline(-var_95, color='red', linestyle='--', linewidth=2, label=f'VaR 95%: {var_95*100:.2f}%')
ax2.set_title('Daily Returns Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('Daily Return')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

✅ OrderBook data structures (bids/asks, price levels)  
✅ Price-time priority matching algorithm  
✅ Order book visualization techniques  
✅ Value at Risk (VaR) calculation using historical simulation  

## Next Steps

Continue to **Notebook #3**: Trading Environment to see how the OrderBook integrates with the RL environment!

---